# 03 · Modelling: is street light availability a predictor of street crime?

Methods follow the module's machine learning seminars:

| Step | Seminar |
|---|---|
| OLS baseline with standardised coefficients | ML1 |
| Random forest regressor | ML2 |
| `GridSearchCV`, training vs CV RMSE, separate final test | Validation |
| SHAP explanations (beeswarm, mapped values) | ML1 |
| **Extension:** spatial `GroupKFold` and a block hold-out test set | Validation seminar's warning about nearby areas |

The whole pipeline runs twice: for **street crime** (lighting-sensitive) and the
**comparison** group. If lighting matters equally for both, the association is
more likely to reflect confounding (e.g. footfall) than a lighting effect.

**Target:** `log1p` of crime per 1,000 residents.
**Inputs:** SLAI, log population density, log commercial points of interest.
Coordinates are deliberately left out, so the model has to rely on area
characteristics that could transfer to new places.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets the notebook import from src/

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src import config

config.FIGURES.mkdir(parents=True, exist_ok=True)
config.TABLES.mkdir(parents=True, exist_ok=True)

import shap
from libpysal import weights
import esda

from src.modelling import run_crime_model

cells = gpd.read_file(config.PROCESSED_CELLS).set_index("cell_id")
urban = cells[cells.study_area.astype(bool)].copy()

urban["log_pop_density"] = np.log1p(urban.pop_density)
features = ["slai", "log_pop_density"]
if urban["n_poi"].notna().any():
    urban["log_poi"] = np.log1p(urban.n_poi)
    features.append("log_poi")

for group in config.CRIME_GROUPS:
    urban[f"log_{group}_rate"] = np.log1p(urban[f"{group}_rate"])

urban = urban.dropna(subset=features + [f"log_{g}_rate" for g in config.CRIME_GROUPS])
print(f"{len(urban):,} cells, {urban.block.nunique()} spatial blocks; features: {features}")

## 1. Fit everything for both crime groups

This runs, for each group: a spatial train–test split (whole 10 km blocks held out),
OLS, random forest grid search under random and spatial CV, the final test, and
out-of-fold predictions for mapping residuals. It can take a few minutes.

In [ ]:
results = {}
for group in config.CRIME_GROUPS:
    print(f"Fitting {group} crime ...")
    results[group] = run_crime_model(urban, f"log_{group}_rate", features, "block", config)
print("Done")

## 2. OLS coefficients

Inputs are standardised, so each coefficient is the change in log crime rate
for a one standard deviation increase, holding the other inputs fixed.

In [ ]:
ols_table = pd.DataFrame({g: r["ols_coefs"] for g, r in results.items()})
ols_table.to_csv(config.TABLES / "ols_standardised_coefficients.csv")
ols_table.round(3)

## 3. Random vs spatial cross-validation

The gap between training RMSE and CV RMSE shows overfitting. The gap between
random and spatial CV RMSE shows how much random folds flatter the model,
because neighbouring cells end up on both sides of the split.

In [ ]:
cv_table = pd.concat({g: r["cv_table"] for g, r in results.items()}, names=["crime group"])
cv_table.to_csv(config.TABLES / "cv_random_vs_spatial.csv")
cv_table

## 4. Final test on held-out spatial blocks

In [ ]:
test_table = pd.concat({g: r["test_table"] for g, r in results.items()}, names=["crime group"])
test_table.to_csv(config.TABLES / "test_scores.csv")
test_table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (group, r) in zip(axes, results.items()):
    test, target = r["test"], f"log_{group}_rate"
    pred = r["final_rf"].predict(test[features])
    ax.scatter(test[target], pred, s=10, alpha=0.4)
    lims = [0, max(test[target].max(), pred.max())]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("Observed (log rate)")
    ax.set_ylabel("Predicted (log rate)")
    ax.set_title(f"{group.capitalize()} crime: held-out blocks")
plt.tight_layout()
plt.savefig(config.FIGURES / "fig04_observed_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. SHAP: how does the forest use street lighting?

`TreeExplainer` is the tree-model version of the explainers used in ML1. It is
exact and fast for random forests. These values explain the fitted model's
predictions; they are not causal effects, and correlated inputs (lighting,
population and shops all cluster in town centres) share credit between them.

In [ ]:
shap_values = {}
for group, r in results.items():
    explainer = shap.TreeExplainer(r["final_rf"])
    shap_values[group] = explainer(urban[features])
    plt.figure()
    shap.plots.beeswarm(shap_values[group], show=False)
    plt.title(f"{group.capitalize()} crime")
    plt.savefig(config.FIGURES / f"fig05_shap_beeswarm_{group}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
importance = pd.DataFrame({
    g: pd.Series(np.abs(sv.values).mean(axis=0), index=features) for g, sv in shap_values.items()
})
importance.to_csv(config.TABLES / "mean_abs_shap.csv")
importance.round(3)

### Where does lighting push predictions up or down?
Red cells: the forest predicts *more* crime because of the cell's lighting level;
blue: *less*. Compare the two crime groups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
slai_idx = features.index("slai")
limit = max(np.abs(sv.values[:, slai_idx]).max() for sv in shap_values.values())
for ax, (group, sv) in zip(axes, shap_values.items()):
    urban[f"shap_slai_{group}"] = sv.values[:, slai_idx]
    cells.plot(ax=ax, color="#f5f5f5", linewidth=0)
    urban.plot(column=f"shap_slai_{group}", ax=ax, cmap="RdBu_r", vmin=-limit, vmax=limit,
               legend=True, linewidth=0, legend_kwds={"shrink": 0.6})
    ax.set_title(f"SHAP value of SLAI: {group} crime")
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(config.FIGURES / "fig06_shap_slai_maps.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Residuals: is there spatial structure the model misses?

Out-of-fold residuals (each cell predicted by a model that never saw its block)
are mapped and tested with Moran's I. Significant positive autocorrelation means
a spatial process the inputs do not capture.

In [ ]:
w = weights.KNN.from_dataframe(urban, k=8)
w.transform = "r"

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
resid_rows = []
for ax, (group, r) in zip(axes, results.items()):
    urban[f"resid_{group}"] = r["oof_resid"]
    mi = esda.Moran(urban[f"resid_{group}"].values, w, permutations=999)
    resid_rows.append({"crime group": group, "Moran's I of residuals": mi.I, "p": mi.p_sim,
                       "out-of-fold RMSE": np.sqrt((r["oof_resid"] ** 2).mean())})
    limit = np.abs(urban[f"resid_{group}"]).quantile(0.98)
    cells.plot(ax=ax, color="#f5f5f5", linewidth=0)
    urban.plot(column=f"resid_{group}", ax=ax, cmap="RdBu_r", vmin=-limit, vmax=limit,
               legend=True, linewidth=0, legend_kwds={"shrink": 0.6})
    ax.set_title(f"Residuals (observed − predicted): {group} crime")
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(config.FIGURES / "fig07_residual_maps.png", dpi=300, bbox_inches="tight")
plt.show()

resid_table = pd.DataFrame(resid_rows)
resid_table.to_csv(config.TABLES / "residual_moran.csv", index=False)
resid_table

## 7. Optional: a shallow decision tree (ML2)

A depth-3 tree gives readable rules for one figure. It is for interpretation
only; the tuned forest is the model to evaluate.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

tree = DecisionTreeRegressor(max_depth=3, random_state=config.RANDOM_STATE)
tree.fit(results["street"]["train"][features], results["street"]["train"]["log_street_rate"])
plt.figure(figsize=(16, 6))
plot_tree(tree, feature_names=features, filled=False, rounded=True, impurity=False, fontsize=9)
plt.savefig(config.FIGURES / "fig08_decision_tree_street.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpreting the results

- Compare SLAI's OLS coefficient, mean |SHAP| and SHAP map **between the two crime groups**.
- Report how much worse spatial CV is than random CV, and whether it picked simpler settings.
- Significant residual autocorrelation means the inputs leave spatial structure unexplained.
- These are **associations** from a cross-sectional design; footfall, policing and the
  June 2023 lighting snapshot are key limitations.